In [3]:
import pandas as pd 
import numpy as np 
import math
import sklearn


In [4]:
df=pd.read_csv("EVChargingStationUsage.csv")
print(df.head(10))
print(df.shape)

                 Station Name          MAC Address           Org Name  \
0  PALO ALTO CA / HAMILTON #1  000D:6F00:015A:9D76  City of Palo Alto   
1  PALO ALTO CA / HAMILTON #1  000D:6F00:015A:9D76  City of Palo Alto   
2  PALO ALTO CA / HAMILTON #1  000D:6F00:015A:9D76  City of Palo Alto   
3  PALO ALTO CA / HAMILTON #1  000D:6F00:015A:9D76  City of Palo Alto   
4  PALO ALTO CA / HAMILTON #1  000D:6F00:015A:9D76  City of Palo Alto   
5  PALO ALTO CA / HAMILTON #1  000D:6F00:015A:9D76  City of Palo Alto   
6  PALO ALTO CA / HAMILTON #1  000D:6F00:015A:9D76  City of Palo Alto   
7  PALO ALTO CA / HAMILTON #1  000D:6F00:015A:9D76  City of Palo Alto   
8  PALO ALTO CA / HAMILTON #1  000D:6F00:015A:9D76  City of Palo Alto   
9  PALO ALTO CA / HAMILTON #1  000D:6F00:015A:9D76  City of Palo Alto   

        Start Date Start Time Zone         End Date End Time Zone  \
0  7/29/2011 20:17             PDT  7/29/2011 23:20           PDT   
1   7/30/2011 0:00             PDT   7/30/2011 0:02       

/tmp/ipykernel_51448/569910306.py:1: DtypeWarning: Columns (29: User ID, 30: County, 32: Model Number) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv("EVChargingStationUsage.csv")


In [5]:
print(df.isnull().sum())
print(df.duplicated().sum())

Station Name                           0
MAC Address                            0
Org Name                               0
Start Date                             0
Start Time Zone                        0
End Date                               0
End Time Zone                          0
Transaction Date (Pacific Time)      209
Total Duration (hh:mm:ss)              0
Charging Time (hh:mm:ss)               0
Energy (kWh)                           0
GHG Savings (kg)                       0
Gasoline Savings (gallons)             0
Port Type                              9
Port Number                            0
Plug Type                              0
EVSE ID                            78948
Address 1                              0
City                                   0
State/Province                         0
Postal Code                            0
Country                                0
Latitude                               0
Longitude                              0
Currency        

In [6]:
df["Start Date"] = pd.to_datetime(df["Start Date"], errors="coerce")
df["End Date"] = pd.to_datetime(df["End Date"], errors="coerce")
total_td = pd.to_timedelta(df['Total Duration (hh:mm:ss)'], errors='coerce')
chg_td = pd.to_timedelta(df['Charging Time (hh:mm:ss)'], errors='coerce')


In [7]:
invalid_dates = ( df["Start Date"].isna()| df["End Date"].isna()| (df["End Date"] <= df["Start Date"]))
print("Invalid timestamp records:", invalid_dates.sum())
display(df.loc[invalid_dates,["Start Date","End Date","Total Duration (hh:mm:ss)","Charging Time (hh:mm:ss)","Energy (kWh)"]].head(20))
print("Negative energy:", (df["Energy (kWh)"] < 0).sum())
print("Zero energy:", (df["Energy (kWh)"] == 0).sum())
print(df["Total Duration (hh:mm:ss)"].value_counts().head()) 

Invalid timestamp records: 83


,Start Date,End Date,Total Duration (hh:mm:ss),Charging Time (hh:mm:ss),Energy (kWh)
390,2011-10-10 15:23:00,2011-10-10 15:23:00,4:34:38,2:37:06,9.155255
15461,2013-10-03 12:21:00,2013-10-03 12:21:00,0:38:10,0:38:04,2.286150
15463,2013-10-03 12:31:00,2013-10-03 12:31:00,2:48:15,2:45:16,3.526191
15464,2013-10-03 12:31:00,2013-10-03 12:31:00,0:55:05,0:54:49,2.805981
15465,2013-10-03 12:41:00,2013-10-03 12:41:00,1:11:00,1:09:09,2.085362
15466,2013-10-03 12:51:00,2013-10-03 12:51:00,0:25:55,0:25:39,0.959668
15469,2013-10-03 13:12:00,2013-10-03 13:12:00,2:32:34,2:21:55,8.072627
15470,2013-10-03 13:32:00,2013-10-03 13:32:00,5:11:19,2:01:04,6.019720
15472,2013-10-03 13:42:00,2013-10-03 13:42:00,1:01:34,1:01:19,3.674747
15474,2013-10-03 13:52:00,2013-10-03 13:52:00,8:46:58,3:59:31,12.199721


Negative energy: 0
Zero energy: 0
Total Duration (hh:mm:ss)
0:01:17    36
1:29:33    36
1:20:51    36
1:05:34    36
0:01:11    35
Name: count, dtype: int64


In [8]:

# Step 1: Calculate duration from start and end timestamps
df["Timestamp_Duration"] = df["End Date"] - df["Start Date"]

# Step 2: Calculate the difference from the recorded total duration
df["Duration_Difference_Minutes"] = (df["Timestamp_Duration"] - total_td).abs().dt.total_seconds() / 60

# Step 3: Identify suspicious records
duration_issues = df[df["Start Date"].isna() | df["End Date"].isna() | (df["End Date"] <= df["Start Date"]) | total_td.isna() | (df["Duration_Difference_Minutes"] > 2)]

# Step 4: Display the results
print("Total records:", len(df))
print("Records needing timestamp investigation:", len(duration_issues))
display(duration_issues[["Station Name", "Start Date", "End Date", "Total Duration (hh:mm:ss)", "Energy (kWh)", "Duration_Difference_Minutes"]].head(20))


Total records: 259415
Records needing timestamp investigation: 494


,Station Name,Start Date,End Date,Total Duration (hh:mm:ss),Energy (kWh),Duration_Difference_Minutes
77,PALO ALTO CA / HAMILTON #1,2011-08-18 11:59:00,2011-08-18 13:42:00,2:24:23,5.014686,41.383333
90,PALO ALTO CA / HAMILTON #2,2011-08-20 14:43:00,2011-08-21 01:44:00,7:25:11,5.268692,215.816667
186,PALO ALTO CA / HIGH #4,2011-09-09 09:33:00,2011-09-09 10:38:00,1:09:46,4.057364,4.766667
275,PALO ALTO CA / HAMILTON #2,2011-09-24 17:40:00,2011-09-24 18:41:00,11:33:33,3.536726,632.550000
390,PALO ALTO CA / BRYANT #2,2011-10-10 15:23:00,2011-10-10 15:23:00,4:34:38,9.155255,274.633333
2103,PALO ALTO CA / HIGH #4,2012-03-10 18:00:00,2012-03-11 09:36:00,14:36:29,10.888565,59.516667
2108,PALO ALTO CA / HAMILTON #1,2012-03-10 22:49:00,2012-03-11 10:34:00,10:44:52,11.885630,60.133333
2372,PALO ALTO CA / HAMILTON #1,2012-03-26 18:23:00,2012-03-26 22:37:00,3:15:06,7.010081,58.900000
2389,PALO ALTO CA / HAMILTON #1,2012-03-27 14:40:00,2012-05-09 12:43:00,22:59:59,0.476540,60423.016667
2431,PALO ALTO CA / BRYANT #1,2012-03-30 08:07:00,2012-03-30 10:26:00,1:10:09,4.234088,68.850000


In [9]:
# Keep records with valid start and end timestamps
valid_time_mask = (df["Start Date"].notna() & df["End Date"].notna() & (df["End Date"] > df["Start Date"]))
df_time = df.loc[valid_time_mask].copy()
print("Original records:", len(df))
print("Records with valid timestamp order:", len(df_time))
print("Records excluded from time-based analysis:", (~valid_time_mask).sum())

Original records: 259415
Records with valid timestamp order: 259332
Records excluded from time-based analysis: 83


In [10]:
duration_review = df.loc[df["Start Date"].notna() & df["End Date"].notna()& (df["End Date"] > df["Start Date"])
                         & (df["Duration_Difference_Minutes"] > 2)].copy()

print("Records with duration discrepancies:", len(duration_review))

Records with duration discrepancies: 411


In [11]:
# Summarize the duration differences
print(duration_review["Duration_Difference_Minutes"].describe(percentiles=[0.50, 0.90, 0.95, 0.99]))

# Display the 20 largest differences
display(duration_review.sort_values("Duration_Difference_Minutes",ascending=False)[
        ["Station Name","Start Date","End Date","Total Duration (hh:mm:ss)","Energy (kWh)","Duration_Difference_Minutes"]
    ].head(20))

count       411.000000
mean       3742.331590
std       15220.212859
min           2.083333
50%         107.866667
90%        5559.733333
95%       11522.091667
99%      100886.953333
max      118437.850000
Name: Duration_Difference_Minutes, dtype: float64


,Station Name,Start Date,End Date,Total Duration (hh:mm:ss),Energy (kWh),Duration_Difference_Minutes
38666,PALO ALTO CA / MPL #6,2015-06-18 13:01:00,2015-09-08 20:24:00,1:25:09,8.338037,118437.850000
38664,PALO ALTO CA / MPL #5,2015-06-18 12:10:00,2015-09-08 20:21:00,2:18:01,13.213822,118432.983333
38606,PALO ALTO CA / MPL #4,2015-06-17 13:04:00,2015-08-27 09:57:00,0:30:13,2.496953,102022.783333
38609,PALO ALTO CA / MPL #4,2015-06-17 14:24:00,2015-08-27 09:57:00,1:25:51,8.336936,101887.150000
38644,PALO ALTO CA / MPL #4,2015-06-18 06:28:00,2015-08-27 09:57:00,1:25:22,2.630938,100923.633333
38673,PALO ALTO CA / MPL #4,2015-06-18 13:55:00,2015-08-27 09:57:00,0:05:10,0.226734,100556.833333
38674,PALO ALTO CA / MPL #4,2015-06-18 14:13:00,2015-08-27 09:57:00,0:15:18,0.554137,100528.700000
39195,PALO ALTO CA / MPL #4,2015-06-30 13:14:00,2015-08-27 09:57:00,0:20:16,1.184045,83302.733333
2389,PALO ALTO CA / HAMILTON #1,2012-03-27 14:40:00,2012-05-09 12:43:00,22:59:59,0.476540,60423.016667
16244,PALO ALTO CA / HAMILTON #2,2013-10-30 17:28:00,2013-12-06 09:45:00,22:59:59,1.738849,51437.016667


In [12]:

# Count the start/end time-zone combinations
print(
    df.groupby(
        ["Start Time Zone", "End Time Zone"],
        dropna=False
    ).size().reset_index(name="Records")
)


  Start Time Zone End Time Zone  Records
0             PDT           PDT   166732
1             PDT           PST       30
2             PDT           UTC        1
3             PST           PDT       58
4             PST           PST    92563
5             UTC           PDT        1
6             UTC           PST       30


In [13]:

# Compare recorded duration with the difference between timestamps
mixed_zone = df[df["Start Time Zone"] != df["End Time Zone"]].copy()

mixed_zone["Timestamp_Duration_Minutes"] = (mixed_zone["End Date"] - mixed_zone["Start Date"]).dt.total_seconds() / 60

mixed_zone["Recorded_Duration_Minutes"] = (total_td.loc[mixed_zone.index].dt.total_seconds() / 60)

mixed_zone["Duration_Difference_Minutes"] = (mixed_zone["Timestamp_Duration_Minutes"]- mixed_zone["Recorded_Duration_Minutes"]).abs()

display(
    mixed_zone[
        ["Station Name","Start Date","Start Time Zone","End Date","End Time Zone","Timestamp_Duration_Minutes",
         "Recorded_Duration_Minutes","Duration_Difference_Minutes"] ].head(20))


,Station Name,Start Date,Start Time Zone,End Date,End Time Zone,Timestamp_Duration_Minutes,Recorded_Duration_Minutes,Duration_Difference_Minutes
2103,PALO ALTO CA / HIGH #4,2012-03-10 18:00:00,PST,2012-03-11 09:36:00,PDT,936.0,876.483333,59.516667
2108,PALO ALTO CA / HAMILTON #1,2012-03-10 22:49:00,PST,2012-03-11 10:34:00,PDT,705.0,644.866667,60.133333
6617,PALO ALTO CA / HAMILTON #2,2012-11-03 22:54:00,PDT,2012-11-04 06:50:00,PST,476.0,536.266667,60.266667
9655,PALO ALTO CA / HAMILTON #2,2013-03-10 00:52:00,PST,2013-03-10 02:57:00,PDT,125.0,65.033333,59.966667
16212,PALO ALTO CA / HAMILTON #1,2013-10-29 16:53:00,UTC,2013-10-29 13:59:00,PDT,-174.0,65.233333,239.233333
16244,PALO ALTO CA / HAMILTON #2,2013-10-30 17:28:00,PDT,2013-12-06 09:45:00,PST,52817.0,1379.983333,51437.016667
19198,PALO ALTO CA / HAMILTON #2,2014-02-10 19:14:00,UTC,2014-02-10 14:17:00,PST,-297.0,182.783333,479.783333
19227,PALO ALTO CA / HAMILTON #2,2014-02-11 19:08:00,UTC,2014-02-11 11:47:00,PST,-441.0,38.983333,479.983333
19228,PALO ALTO CA / HAMILTON #2,2014-02-11 19:56:00,UTC,2014-02-11 14:15:00,PST,-341.0,139.350000,480.350000
19231,PALO ALTO CA / HIGH #4,2014-02-11 22:14:00,UTC,2014-02-11 16:21:00,PST,-353.0,126.466667,479.466667


In [14]:
zone_map = {
    "PST": "Etc/GMT+8",
    "PDT": "Etc/GMT+7",
    "UTC": "UTC"
}

# Create new UTC columns; do not overwrite the original timestamps
df["Start_UTC"] = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns, UTC]")
df["End_UTC"] = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns, UTC]")

# Convert each start timestamp according to its own label
for label, timezone in zone_map.items():
    mask = ( df["Start Time Zone"].eq(label) & df["Start Date"].notna())
    df.loc[mask, "Start_UTC"] = ( df.loc[mask, "Start Date"] .dt.tz_localize(timezone).dt.tz_convert("UTC"))

# Convert each end timestamp according to its own label
for label, timezone in zone_map.items():
    mask = ( df["End Time Zone"].eq(label) & df["End Date"].notna())
    df.loc[mask, "End_UTC"] = (df.loc[mask, "End Date"].dt.tz_localize(timezone).dt.tz_convert("UTC"))

# Calculate elapsed time using the converted timestamps
df["Elapsed_UTC_Minutes"] = (df["End_UTC"] - df["Start_UTC"]).dt.total_seconds() / 60

# Check the results
display(
    df[["Start Date", "Start Time Zone", "Start_UTC","End Date", "End Time Zone", "End_UTC",
            "Elapsed_UTC_Minutes","Total Duration (hh:mm:ss)"]].head(10))


,Start Date,Start Time Zone,Start_UTC,End Date,End Time Zone,End_UTC,Elapsed_UTC_Minutes,Total Duration (hh:mm:ss)
0,2011-07-29 20:17:00,PDT,2011-07-30 03:17:00+00:00,2011-07-29 23:20:00,PDT,2011-07-30 06:20:00+00:00,183.0,3:03:32
1,2011-07-30 00:00:00,PDT,2011-07-30 07:00:00+00:00,2011-07-30 00:02:00,PDT,2011-07-30 07:02:00+00:00,2.0,0:02:06
2,2011-07-30 08:16:00,PDT,2011-07-30 15:16:00+00:00,2011-07-30 12:34:00,PDT,2011-07-30 19:34:00+00:00,258.0,4:17:32
3,2011-07-30 14:51:00,PDT,2011-07-30 21:51:00+00:00,2011-07-30 16:55:00,PDT,2011-07-30 23:55:00+00:00,124.0,2:03:24
4,2011-07-30 18:51:00,PDT,2011-07-31 01:51:00+00:00,2011-07-30 20:03:00,PDT,2011-07-31 03:03:00+00:00,72.0,1:11:24
5,2011-07-30 21:38:00,PDT,2011-07-31 04:38:00+00:00,2011-07-31 01:30:00,PDT,2011-07-31 08:30:00+00:00,232.0,3:52:13
6,2011-07-31 04:33:00,PDT,2011-07-31 11:33:00+00:00,2011-07-31 10:40:00,PDT,2011-07-31 17:40:00+00:00,367.0,6:06:19
7,2011-07-31 12:25:00,PDT,2011-07-31 19:25:00+00:00,2011-07-31 13:35:00,PDT,2011-07-31 20:35:00+00:00,70.0,1:09:54
8,2011-07-31 17:41:00,PDT,2011-08-01 00:41:00+00:00,2011-07-31 22:31:00,PDT,2011-08-01 05:31:00+00:00,290.0,4:49:46
9,2011-08-01 19:11:00,PDT,2011-08-02 02:11:00+00:00,2011-08-01 22:34:00,PDT,2011-08-02 05:34:00+00:00,203.0,3:23:34


In [15]:

# Convert recorded duration to minutes
recorded_minutes = total_td.dt.total_seconds() / 60

# Calculate elapsed time using UTC timestamps
df["Elapsed_UTC_Minutes"] = (
    df["End_UTC"] - df["Start_UTC"]
).dt.total_seconds() / 60

# Compare elapsed time with recorded duration
df["UTC_Duration_Difference_Minutes"] = (
    df["Elapsed_UTC_Minutes"] - recorded_minutes
).abs()

# Summarize the differences
print("Total records:", len(df))
print(
    "Records with difference over 2 minutes:",
    (df["UTC_Duration_Difference_Minutes"] > 2).sum()
)
print(
    "Records with negative UTC duration:",
    (df["Elapsed_UTC_Minutes"] <= 0).sum()
)

display(
    df.sort_values(
        "UTC_Duration_Difference_Minutes",
        ascending=False
    )[
        [
            "Start Date",
            "Start Time Zone",
            "Start_UTC",
            "End Date",
            "End Time Zone",
            "End_UTC",
            "Total Duration (hh:mm:ss)",
            "Elapsed_UTC_Minutes",
            "UTC_Duration_Difference_Minutes"
        ]
    ].head(15)
)


Total records: 259415
Records with difference over 2 minutes: 339
Records with negative UTC duration: 12


,Start Date,Start Time Zone,Start_UTC,End Date,End Time Zone,End_UTC,Total Duration (hh:mm:ss),Elapsed_UTC_Minutes,UTC_Duration_Difference_Minutes
38666,2015-06-18 13:01:00,PDT,2015-06-18 20:01:00+00:00,2015-09-08 20:24:00,PDT,2015-09-09 03:24:00+00:00,1:25:09,118523.0,118437.850000
38664,2015-06-18 12:10:00,PDT,2015-06-18 19:10:00+00:00,2015-09-08 20:21:00,PDT,2015-09-09 03:21:00+00:00,2:18:01,118571.0,118432.983333
38606,2015-06-17 13:04:00,PDT,2015-06-17 20:04:00+00:00,2015-08-27 09:57:00,PDT,2015-08-27 16:57:00+00:00,0:30:13,102053.0,102022.783333
38609,2015-06-17 14:24:00,PDT,2015-06-17 21:24:00+00:00,2015-08-27 09:57:00,PDT,2015-08-27 16:57:00+00:00,1:25:51,101973.0,101887.150000
38644,2015-06-18 06:28:00,PDT,2015-06-18 13:28:00+00:00,2015-08-27 09:57:00,PDT,2015-08-27 16:57:00+00:00,1:25:22,101009.0,100923.633333
38673,2015-06-18 13:55:00,PDT,2015-06-18 20:55:00+00:00,2015-08-27 09:57:00,PDT,2015-08-27 16:57:00+00:00,0:05:10,100562.0,100556.833333
38674,2015-06-18 14:13:00,PDT,2015-06-18 21:13:00+00:00,2015-08-27 09:57:00,PDT,2015-08-27 16:57:00+00:00,0:15:18,100544.0,100528.700000
39195,2015-06-30 13:14:00,PDT,2015-06-30 20:14:00+00:00,2015-08-27 09:57:00,PDT,2015-08-27 16:57:00+00:00,0:20:16,83323.0,83302.733333
2389,2012-03-27 14:40:00,PDT,2012-03-27 21:40:00+00:00,2012-05-09 12:43:00,PDT,2012-05-09 19:43:00+00:00,22:59:59,61803.0,60423.016667
16244,2013-10-30 17:28:00,PDT,2013-10-31 00:28:00+00:00,2013-12-06 09:45:00,PST,2013-12-06 17:45:00+00:00,22:59:59,52877.0,51497.016667


In [16]:

# Find records with zero or negative UTC duration
nonpositive_utc = df[
    df["Elapsed_UTC_Minutes"] <= 0
].copy()

# Select only the important columns
columns_to_show = [
    "Station Name",
    "Start Date",
    "Start Time Zone",
    "End Date",
    "End Time Zone",
    "Total Duration (hh:mm:ss)",
    "Elapsed_UTC_Minutes"
]

# Display as a readable table
display(
    nonpositive_utc[columns_to_show]
    .reset_index(drop=True)
)

print("Number of records to investigate:", len(nonpositive_utc))


,Station Name,Start Date,Start Time Zone,End Date,End Time Zone,Total Duration (hh:mm:ss),Elapsed_UTC_Minutes
0,PALO ALTO CA / BRYANT #2,2011-10-10 15:23:00,PDT,2011-10-10 15:23:00,PDT,4:34:38,0.0
1,PALO ALTO CA / BRYANT #2,2013-10-03 12:21:00,PDT,2013-10-03 12:21:00,PDT,0:38:10,0.0
2,PALO ALTO CA / BRYANT #2,2013-10-03 12:31:00,PDT,2013-10-03 12:31:00,PDT,2:48:15,0.0
3,PALO ALTO CA / BRYANT #2,2013-10-03 12:31:00,PDT,2013-10-03 12:31:00,PDT,0:55:05,0.0
4,PALO ALTO CA / BRYANT #2,2013-10-03 12:41:00,PDT,2013-10-03 12:41:00,PDT,1:11:00,0.0
5,PALO ALTO CA / BRYANT #2,2013-10-03 12:51:00,PDT,2013-10-03 12:51:00,PDT,0:25:55,0.0
6,PALO ALTO CA / BRYANT #2,2013-10-03 13:12:00,PDT,2013-10-03 13:12:00,PDT,2:32:34,0.0
7,PALO ALTO CA / BRYANT #2,2013-10-03 13:32:00,PDT,2013-10-03 13:32:00,PDT,5:11:19,0.0
8,PALO ALTO CA / BRYANT #2,2013-10-03 13:42:00,PDT,2013-10-03 13:42:00,PDT,1:01:34,0.0
9,PALO ALTO CA / BRYANT #2,2013-10-03 13:52:00,PDT,2013-10-03 13:52:00,PDT,8:46:58,0.0


Number of records to investigate: 12


In [17]:

# Step 1: Ensure energy is numeric
df["Energy (kWh)"] = pd.to_numeric(
    df["Energy (kWh)"], errors="coerce"
)

# Step 2: Identify records suitable for initial time-based analysis
time_quality_mask = (
    df["Start_UTC"].notna()
    & df["End_UTC"].notna()
    & (df["Elapsed_UTC_Minutes"] > 0)
    & (df["UTC_Duration_Difference_Minutes"] <= 2)
    & df["Station Name"].notna()
    & df["Energy (kWh)"].notna()
)

# Step 3: Create a separate analysis dataset
df_time_clean = df.loc[time_quality_mask].copy()

# Step 4: Keep the records requiring review separately
df_time_review = df.loc[~time_quality_mask].copy()

# Step 5: Check the results
print("Original records:", len(df))
print("Records for initial time-based analysis:", len(df_time_clean))
print("Records requiring review:", len(df_time_review))

print("\nReview reasons:")
print("Missing UTC timestamps:",
      (df["Start_UTC"].isna() | df["End_UTC"].isna()).sum())
print("Zero or negative elapsed duration:",
      (df["Elapsed_UTC_Minutes"] <= 0).sum())
print("Duration difference over 2 minutes:",
      (df["UTC_Duration_Difference_Minutes"] > 2).sum())


Original records: 259415
Records for initial time-based analysis: 259036
Records requiring review: 379

Review reasons:
Missing UTC timestamps: 40
Zero or negative elapsed duration: 12
Duration difference over 2 minutes: 339


In [18]:

# Create the hour for each charging session's start time
df_time_clean["Hour"] = df_time_clean["Start_UTC"].dt.floor("h")

# Calculate total energy by station and hour
hourly_energy = (
    df_time_clean
    .groupby(["Station Name", "Hour"])["Energy (kWh)"]
    .sum()
    .reset_index()
    .sort_values(["Station Name", "Hour"])
)

# Check the results
print("Hourly observations:", len(hourly_energy))
print("Number of stations:", hourly_energy["Station Name"].nunique())

display(hourly_energy.head(10))


Hourly observations: 234691
Number of stations: 47


,Station Name,Hour,Energy (kWh)
0,PALO ALTO CA / BRYANT # 1,2020-09-06 18:00:00+00:00,5.355
1,PALO ALTO CA / BRYANT # 1,2020-09-07 19:00:00+00:00,5.297
2,PALO ALTO CA / BRYANT # 1,2020-10-13 20:00:00+00:00,4.361
3,PALO ALTO CA / BRYANT # 1,2020-10-14 16:00:00+00:00,42.064
4,PALO ALTO CA / BRYANT # 1,2020-10-17 23:00:00+00:00,5.430
5,PALO ALTO CA / BRYANT # 1,2020-10-18 01:00:00+00:00,10.662
6,PALO ALTO CA / BRYANT # 1,2020-10-18 07:00:00+00:00,2.830
7,PALO ALTO CA / BRYANT # 1,2020-10-19 20:00:00+00:00,1.867
8,PALO ALTO CA / BRYANT # 1,2020-10-20 20:00:00+00:00,6.946
9,PALO ALTO CA / BRYANT # 1,2020-10-21 07:00:00+00:00,1.480


In [19]:

# Select one station to test the process
station_name = hourly_energy["Station Name"].iloc[0]

station_data = (
    hourly_energy[
        hourly_energy["Station Name"] == station_name
    ]
    .set_index("Hour")["Energy (kWh)"]
    .sort_index()
)

# Create a continuous hourly timeline
full_hours = pd.date_range(
    start=station_data.index.min(),
    end=station_data.index.max(),
    freq="h"
)

# Add missing hours as zero recorded energy
station_hourly = station_data.reindex(full_hours, fill_value=0)

print("Station:", station_name)
print("First hour:", station_hourly.index.min())
print("Last hour:", station_hourly.index.max())
print("Total hourly time points:", len(station_hourly))
print("Hours with zero recorded energy:", (station_hourly == 0).sum())

display(station_hourly.head(24).to_frame("Energy (kWh)"))


Station: PALO ALTO CA / BRYANT # 1
First hour: 2020-09-06 18:00:00+00:00
Last hour: 2020-12-25 02:00:00+00:00
Total hourly time points: 2625
Hours with zero recorded energy: 2612


,Energy (kWh)
2020-09-06 18:00:00+00:00,5.355
2020-09-06 19:00:00+00:00,0.000
2020-09-06 20:00:00+00:00,0.000
2020-09-06 21:00:00+00:00,0.000
2020-09-06 22:00:00+00:00,0.000
2020-09-06 23:00:00+00:00,0.000
2020-09-07 00:00:00+00:00,0.000
2020-09-07 01:00:00+00:00,0.000
2020-09-07 02:00:00+00:00,0.000
2020-09-07 03:00:00+00:00,0.000


In [20]:

# Check how many records belong to each station name
station_counts = (
    df_time_clean["Station Name"]
    .value_counts()
    .rename_axis("Station Name")
    .reset_index(name="Session_Count")
)

print("Number of station-name labels:", len(station_counts))

display(station_counts.head(20))

# Check the date range for each station label
station_summary = (
    df_time_clean.groupby("Station Name")
    .agg(
        Sessions=("Energy (kWh)", "size"),
        First_Session=("Start_UTC", "min"),
        Last_Session=("Start_UTC", "max")
    )
    .sort_values("Sessions")
)

display(station_summary.head(15))


Number of station-name labels: 47


,Station Name,Session_Count
0,PALO ALTO CA / HAMILTON #2,23691
1,PALO ALTO CA / HAMILTON #1,14846
2,PALO ALTO CA / WEBSTER #1,14416
3,PALO ALTO CA / BRYANT #2,14305
4,PALO ALTO CA / HIGH #4,13858
5,PALO ALTO CA / BRYANT #1,13397
6,PALO ALTO CA / WEBSTER #2,12660
7,PALO ALTO CA / CAMBRIDGE #1,11160
8,PALO ALTO CA / HIGH #1,10749
9,PALO ALTO CA / WEBSTER #3,10477


,Sessions,First_Session,Last_Session
Station Name,,,
PALO ALTO CA / SHERMAN 11,1,2020-12-29 17:01:00+00:00,2020-12-29 17:01:00+00:00
PALO ALTO CA / SHERMAN 15,1,2020-12-26 21:50:00+00:00,2020-12-26 21:50:00+00:00
PALO ALTO CA / SHERMAN 17,1,2020-12-29 17:03:00+00:00,2020-12-29 17:03:00+00:00
PALO ALTO CA / SHERMAN 5,1,2020-12-26 00:44:00+00:00,2020-12-26 00:44:00+00:00
PALO ALTO CA / SHERMAN 9,1,2020-12-13 20:44:00+00:00,2020-12-13 20:44:00+00:00
PALO ALTO CA / SHERMAN 8,2,2020-12-13 22:35:00+00:00,2020-12-19 04:19:00+00:00
PALO ALTO CA / SHERMAN 14,2,2020-12-18 19:38:00+00:00,2020-12-27 22:34:00+00:00
PALO ALTO CA / SHERMAN 6,3,2020-12-09 22:46:00+00:00,2020-12-31 19:04:00+00:00
PALO ALTO CA / SHERMAN 4,4,2020-12-14 23:10:00+00:00,2020-12-27 19:37:00+00:00


In [21]:

# Create a normalized label for checking only
station_check = df_time_clean[["Station Name"]].drop_duplicates().copy()

station_check["Normalized_Name"] = (
    station_check["Station Name"]
    .str.upper()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.replace(r"\s*#\s*", " #", regex=True)
)

# Find normalized names that correspond to multiple original labels
name_variations = (
    station_check.groupby("Normalized_Name")["Station Name"]
    .agg(list)
    .reset_index(name="Original_Names")
)

name_variations["Number_of_Variations"] = (
    name_variations["Original_Names"].str.len()
)

name_variations = name_variations[
    name_variations["Number_of_Variations"] > 1
]

display(name_variations)


,Normalized_Name,Original_Names,Number_of_Variations
0,PALO ALTO CA / BRYANT #1,"[PALO ALTO CA / BRYANT #1, PALO ALTO CA / BRYA...",2


In [22]:

print(
    df_time_clean[
        df_time_clean["Station Name"].str.contains(
            "BRYANT", case=False, na=False
        )
    ]["Station Name"].value_counts()
)


Station Name
PALO ALTO CA / BRYANT #2     14305
PALO ALTO CA / BRYANT #1     13397
PALO ALTO CA / BRYANT #6      5856
PALO ALTO CA / BRYANT #5      5329
PALO ALTO CA / BRYANT #3      4464
PALO ALTO CA / BRYANT #4      3596
PALO ALTO CA / BRYANT # 1       13
Name: count, dtype: int64


In [23]:

# Correct the confirmed Bryant #1 naming variation
df_time_clean["Station Name"] = (
    df_time_clean["Station Name"]
    .replace({
        "PALO ALTO CA / BRYANT # 1":
        "PALO ALTO CA / BRYANT #1"
    })
)

# Verify the result
print(
    df_time_clean[
        df_time_clean["Station Name"].str.contains(
            "BRYANT", case=False, na=False
        )
    ]["Station Name"].value_counts()
)


Station Name
PALO ALTO CA / BRYANT #2    14305
PALO ALTO CA / BRYANT #1    13410
PALO ALTO CA / BRYANT #6     5856
PALO ALTO CA / BRYANT #5     5329
PALO ALTO CA / BRYANT #3     4464
PALO ALTO CA / BRYANT #4     3596
Name: count, dtype: int64


In [24]:

# Create hourly energy for all stations
df_time_clean["Hour"] = df_time_clean["Start_UTC"].dt.floor("h")

hourly_energy = (
    df_time_clean
    .groupby(["Station Name", "Hour"])["Energy (kWh)"]
    .sum()
    .reset_index()
    .sort_values(["Station Name", "Hour"])
)

# Display overall results
print("Hourly observations:", len(hourly_energy))
print("Number of stations:", hourly_energy["Station Name"].nunique())

# Display the first 50 hourly records
display(hourly_energy.head(50))

# Display a summary for ALL stations
station_summary = (
    hourly_energy
    .groupby("Station Name")
    .agg(
        Hourly_Observations=("Hour", "count"),
        Total_Energy_kWh=("Energy (kWh)", "sum"),
        First_Hour=("Hour", "min"),
        Last_Hour=("Hour", "max")
    )
    .reset_index()
    .sort_values("Total_Energy_kWh", ascending=False)
)

print("\nSummary for all stations:")
display(station_summary)


Hourly observations: 234691
Number of stations: 46


,Station Name,Hour,Energy (kWh)
0,PALO ALTO CA / BRYANT #1,2011-10-13 19:00:00+00:00,1.630508
1,PALO ALTO CA / BRYANT #1,2011-10-14 01:00:00+00:00,6.631544
2,PALO ALTO CA / BRYANT #1,2011-10-14 21:00:00+00:00,6.259466
3,PALO ALTO CA / BRYANT #1,2011-10-15 19:00:00+00:00,4.622894
4,PALO ALTO CA / BRYANT #1,2011-10-17 20:00:00+00:00,4.023361
5,PALO ALTO CA / BRYANT #1,2011-10-18 20:00:00+00:00,15.474944
6,PALO ALTO CA / BRYANT #1,2011-10-20 00:00:00+00:00,6.762254
7,PALO ALTO CA / BRYANT #1,2011-10-20 20:00:00+00:00,6.073878
8,PALO ALTO CA / BRYANT #1,2011-10-21 16:00:00+00:00,1.482446
9,PALO ALTO CA / BRYANT #1,2011-10-21 19:00:00+00:00,5.135938



Summary for all stations:


,Station Name,Hourly_Observations,Total_Energy_kWh,First_Hour,Last_Hour
12,PALO ALTO CA / HAMILTON #2,20666,191370.913093,2011-08-02 19:00:00+00:00,2020-11-04 12:00:00+00:00
43,PALO ALTO CA / WEBSTER #1,12343,138899.008993,2015-07-29 20:00:00+00:00,2021-01-01 01:00:00+00:00
44,PALO ALTO CA / WEBSTER #2,10820,125674.023719,2015-07-29 23:00:00+00:00,2020-12-31 01:00:00+00:00
11,PALO ALTO CA / HAMILTON #1,14287,113457.607750,2011-07-30 03:00:00+00:00,2020-11-03 21:00:00+00:00
16,PALO ALTO CA / HIGH #4,12835,111531.307913,2011-09-03 03:00:00+00:00,2020-12-07 00:00:00+00:00
1,PALO ALTO CA / BRYANT #2,13455,106723.143087,2011-10-11 02:00:00+00:00,2020-07-01 20:00:00+00:00
0,PALO ALTO CA / BRYANT #1,12602,102147.681243,2011-10-13 19:00:00+00:00,2020-12-25 02:00:00+00:00
45,PALO ALTO CA / WEBSTER #3,9103,98512.988563,2016-01-14 21:00:00+00:00,2020-12-31 22:00:00+00:00
13,PALO ALTO CA / HIGH #1,9208,98359.504769,2015-07-29 22:00:00+00:00,2020-12-31 17:00:00+00:00
39,PALO ALTO CA / TED THOMPSON #1,8505,91144.395604,2016-01-14 18:00:00+00:00,2020-10-11 17:00:00+00:00


In [25]:

# Extract hour of day from the UTC timestamp
hourly_energy["Hour_Of_Day"] = hourly_energy["Hour"].dt.hour

# Calculate total energy and number of active station-hours for each hour
demand_by_hour = (
    hourly_energy.groupby("Hour_Of_Day")
    .agg(
        Total_Energy_kWh=("Energy (kWh)", "sum"),
        Active_Station_Hours=("Station Name", "count")
    )
    .reset_index()
)

print("Energy and activity by hour of day:")
display(demand_by_hour)

# Plot total energy by hour of day
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.bar(
    demand_by_hour["Hour_Of_Day"],
    demand_by_hour["Total_Energy_kWh"]
)
plt.title("Total Recorded Energy by Hour of Day (UTC)")
plt.xlabel("Hour of Day (UTC)")
plt.ylabel("Total Energy (kWh)")
plt.xticks(range(24))
plt.tight_layout()
plt.show()


Energy and activity by hour of day:


,Hour_Of_Day,Total_Energy_kWh,Active_Station_Hours
0,0,128181.050497,15531
1,1,144187.108215,16902
2,2,124057.855491,14380
3,3,84188.143986,9325
4,4,65212.446709,5940
5,5,47034.777663,3625
6,6,26916.252887,1838
7,7,12396.197858,745
8,8,4910.734275,296
9,9,2631.348001,163


ModuleNotFoundError: No module named 'matplotlib'

In [ ]:

# Calculate total energy for each station
station_energy = (hourly_energy.groupby("Station Name")["Energy (kWh)"].sum().sort_values(ascending=False))

# Select the top 15 stations
top_15 = station_energy.head(15).sort_values()

# Plot with station names
plt.figure(figsize=(12, 7))
top_15.plot(kind="barh")

plt.title("Top 15 Stations by Total Recorded Energy")
plt.xlabel("Total Energy (kWh)")
plt.ylabel("Station Name")
plt.tight_layout()
plt.show()


In [ ]:

# Select the five stations with the highest total energy
top_5_stations = station_energy.head(5).index

# Extract hour of day
hourly_energy["Hour_Of_Day"] = hourly_energy["Hour"].dt.hour

# Calculate energy by station and hour of day
station_hour_pattern = (
    hourly_energy[hourly_energy["Station Name"].isin(top_5_stations)].groupby(["Station Name", "Hour_Of_Day"])["Energy (kWh)"].sum().reset_index())

# Plot each station as a separate line
plt.figure(figsize=(13, 6))
for station in top_5_stations:
    station_data = station_hour_pattern[station_hour_pattern["Station Name"] == station]
    plt.plot( station_data["Hour_Of_Day"],station_data["Energy (kWh)"],marker="o",label=station)

plt.title("Hourly Session-Start Energy by Station (Top 5)")
plt.xlabel("Hour of Day (UTC)")
plt.ylabel("Total Energy (kWh)")
plt.xticks(range(24))
plt.legend(title="Station Name", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
print("Dataset shape:", df_time_clean.shape)

column_check = pd.DataFrame({"Column": df_time_clean.columns,"Data_Type": df_time_clean.dtypes.astype(str).values,
    "Non_Null_Count": df_time_clean.notna().sum().values,"Missing_Count": df_time_clean.isna().sum().values,
    "Missing_Percentage": (df_time_clean.isna().mean().values * 100).round(2)})

display(column_check)

In [ ]:
# Sessions with exactly 0 minutes
zero_minute = df_time_clean[
    df_time_clean["Elapsed_UTC_Minutes"] == 0
]

# Sessions lasting more than 0 but less than 2 minutes
under_2_minutes = df_time_clean[
    (df_time_clean["Elapsed_UTC_Minutes"] > 0) &
    (df_time_clean["Elapsed_UTC_Minutes"] < 2)
]

# All sessions lasting less than 2 minutes, including zero
less_than_2 = df_time_clean[
    df_time_clean["Elapsed_UTC_Minutes"] < 2
]

print("Zero-minute sessions:", len(zero_minute))
print("Sessions between 0 and 2 minutes:", len(under_2_minutes))
print("Total sessions below 2 minutes:", len(less_than_2))

In [ ]:
evse_usage = (
    df_time_clean
    .dropna(subset=["EVSE ID"])
    .groupby("EVSE ID")
    .size()
    .reset_index(name="Total Sessions")
    .sort_values("Total Sessions", ascending=True)
)
display(evse_usage)